In [1]:
from gliclass.data_processing import GLiClassDataset
from transformers import AutoModel, AutoConfig
from gliclass.config import GLiClassModelConfig
from gliclass.model import GLiClassModel, GLiClassBiEncoder, GLiClassAudio
from transformers import AutoConfig, AutoTokenizer
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor
import torchaudio, torch

/home/werent4/GLiClass/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
audiocfg = AutoConfig.from_pretrained("facebook/wav2vec2-base-960h")
deberta_cfg = AutoConfig.from_pretrained("microsoft/deberta-v3-small")
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")
audio_feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    "facebook/wav2vec2-base-960h"
)


/home/werent4/GLiClass/.venv/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [3]:
glicalss_config = GLiClassModelConfig(
    encoder_config=deberta_cfg,
    encoder_model="microsoft/deberta-v3-small",
    audio_model_name="facebook/wav2vec2-base-960h",
    audio_model_config=audiocfg,
    class_token_index=len(tokenizer),
    text_token_index=len(tokenizer)+1,
    audio_token_index=len(tokenizer)+2, 
    pooling_strategy="first",
    scorer_type="simple",
    use_lstm=False,
    focal_loss_alpha=-1,
    focal_loss_gamma=-1,
    contrastive_loss_coef=0.0,
    normalize_features=False,
    extract_text_features=False,
    architecture_type='audio-encoder',
    prompt_first=True,
    squeeze_layers=False,
    shuffle_labels=True
)

model = GLiClassModel(glicalss_config, from_pretrained=False)
new_words = ["<<LABEL>>", "<<SEP>>", "<<AUDIO>>"]
tokenizer.add_tokens(new_words, special_tokens=True)
model.resize_token_embeddings(len(tokenizer), None)

Embedding(128004, 768, padding_idx=0)

In [4]:
train_data = [
  {"audio": "audio.wav", "all_labels": ["sports", "science", "business"], "true_labels": ["sports"]},
]
audio_raw, sample_rate = torchaudio.load("audio.wav")
features = audio_feature_extractor(audio_raw, sampling_rate=sample_rate, return_tensors="pt")

In [5]:
train_dataset = GLiClassDataset(train_data, tokenizer, 1024, 
                                'multi_label_classification', "audio-encoder", 
                                True, labels_tokenizer=tokenizer)
exmpl = train_dataset[0]

Total labels:  3
Audio bi-encoder


In [6]:
exmpl

{'input_ids': [1, 128001, 1948, 128001, 1693, 128001, 460, 128002, 128003, 2], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': tensor([1., 0., 0.]), 'labels_text': ['sports', 'science', 'business'], 'input_audio': 'audio.wav'}

In [7]:
tokenizer.decode(exmpl['input_ids'])

'[CLS]<<LABEL>> sports<<LABEL>> science<<LABEL>> business<<SEP>><<AUDIO>>[SEP]'

In [8]:
print("input_ids type:", type(exmpl['input_ids']))
print("attention_mask type:", type(exmpl['attention_mask']))
print("labels type:", type(exmpl['labels']))

input_ids type: <class 'list'>
attention_mask type: <class 'list'>
labels type: <class 'torch.Tensor'>


In [9]:
input_ids = torch.tensor(exmpl['input_ids']).unsqueeze(0)  
attention_mask = torch.tensor(exmpl['attention_mask']).unsqueeze(0)  
labels = torch.tensor(exmpl['labels']).unsqueeze(0)  

/tmp/ipykernel_312178/2331616996.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(exmpl['labels']).unsqueeze(0)


In [10]:
exmpl['labels']

tensor([1., 0., 0.])

In [11]:
print("Labels shape:", labels.shape)
print("Labels:", labels)
print("Problem type:", model.config.problem_type)

Labels shape: torch.Size([1, 3])
Labels: tensor([[1., 0., 0.]])
Problem type: None


In [12]:
model(input_ids, attention_mask, features["input_values"], labels=labels)

tensor([8])


GLiClassOutput(loss=tensor(0.7875, grad_fn=<NegBackward0>), logits=tensor([[8.6143, 6.6367, 8.6722]], grad_fn=<ViewBackward0>), hidden_states=None, attentions=None, text_embeddings=tensor([[-1.2394e-01,  1.4817e-01, -4.9551e-01,  2.1445e-01,  1.9457e-01,
         -2.6612e-01, -1.9701e-01, -3.8292e-01,  3.2027e-01, -2.8481e-01,
          3.4388e-01, -0.0000e+00,  9.5565e-02, -1.3620e-01,  3.9522e-01,
         -4.3771e-01, -7.9780e-02,  1.2656e-01, -2.4841e-01, -1.3614e-01,
          3.9507e-02,  3.1015e-01, -4.4022e-01,  1.5950e-01, -9.9297e-02,
          8.8780e-02,  2.3152e-01, -1.7732e-01,  8.0774e-01, -4.0525e-01,
         -5.6045e-01,  4.9860e-01, -4.0380e-01,  1.9270e-01,  1.4664e-01,
         -7.4296e-02, -4.8993e-01, -2.6911e-01, -3.6759e-01,  1.0737e-01,
         -3.7000e-02, -3.2669e-01, -1.8779e-01,  6.5797e-01,  5.3591e-01,
         -3.3322e-01, -1.1549e+00, -3.7330e-01, -2.0866e-01,  7.5672e-01,
          2.0037e-02,  7.2838e-01, -2.4753e-01, -3.7955e-01,  2.0572e-01,
     